# Custom Tools and Voice Generation

The podcaster crew can research, report, and script. Now we close the loop: **topic -> research -> report -> script -> audio**.

You built a tool from scratch in the first notebook (Tavily + JSON schema). CrewAI simplifies this with the `@tool` decorator — same concept, less boilerplate.

| Step | Concept | What It Unlocks |
|------|---------|----------------|
| 1 | **@tool Decorator** | Turn a Python function into an agent-callable tool |
| 2 | **File Tools** | Let agents read and write workflow artifacts |
| 3 | **Gemini TTS Tool** | Generate multi-speaker podcast audio |
| 4 | **Full Pipeline** | Research, report, script, and audio in one crew run |
| 5 | **Dynamic Inputs** | Reuse the workflow for any topic |


In [ ]:
# !pip install -q "crewai[tools]" python-dotenv google-genai pydantic

import base64
import datetime
import os
import wave
from pathlib import Path
from typing import Any, Tuple

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tasks.task_output import TaskOutput
from crewai.tools import tool
from crewai_tools import EXASearchTool, FileReadTool, FileWriterTool
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)


---

## Step 1: The @tool Decorator

In the first notebook you registered a tool by writing a JSON schema and a dispatch function. CrewAI replaces that with `@tool` — decorate any function and the agent can call it.


In [ ]:
@tool("Create episode filename")
def create_episode_filename(topic: str) -> str:
    """Create a safe filename prefix for a podcast episode topic."""
    safe_topic = topic.lower().replace(" ", "-").replace("/", "-")
    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    return f"{safe_topic}-{timestamp}"


print(create_episode_filename.run("AI LLMs"))
print(create_episode_filename.description)


The decorator extracts the function name, docstring, and type hints to build the schema the agent sees. No manual JSON needed.

---

## Step 2: Built-In File Tools

CrewAI ships pre-built tools for common operations. `FileWriterTool` and `FileReadTool` let agents persist and retrieve artifacts during a run.


In [ ]:
search_tool = EXASearchTool()
file_writer_tool = FileWriterTool()
file_read_tool = FileReadTool()

print(f"Search: {search_tool.name}")
print(f"Writer: {file_writer_tool.name}")
print(f"Reader: {file_read_tool.name}")


---

## Step 3: Build the Gemini Voice Tool

This is a real custom tool: it takes a podcast script with Joe/Jane speaker labels and uses Gemini 2.5 Flash TTS to generate a multi-speaker `.wav` file.


In [ ]:
def write_wave_file(filename: str, pcm: bytes, channels: int = 1, rate: int = 24000, sample_width: int = 2) -> None:
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)


@tool
def gemini_voice_tool(script: str) -> str:
    """Generate a multi-speaker podcast audio file from a Joe/Jane script."""
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"))

    response = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=script,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=[
                        types.SpeakerVoiceConfig(
                            speaker="Joe",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
                            ),
                        ),
                        types.SpeakerVoiceConfig(
                            speaker="Jane",
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
                            ),
                        ),
                    ]
                )
            ),
        ),
    )

    inline_audio = response.candidates[0].content.parts[0].inline_data.data
    audio_bytes = base64.b64decode(inline_audio) if isinstance(inline_audio, str) else bytes(inline_audio)
    filename = outputs_dir / f"podcast-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.wav"
    write_wave_file(str(filename), audio_bytes)
    return str(filename)


### Test the Voice Tool Directly

Run this to confirm your Gemini key and audio pipeline work before wiring it into an agent.


In [ ]:
sample_script = (
    "Joe: Welcome back to the show.\n"
    "Jane: Today we are testing our new AI podcast voice pipeline.\n"
    "Joe: If this works, we can turn any research report into audio.\n"
    "Jane: That is the whole production workflow in one notebook."
)

# Uncomment when GEMINI_API_KEY is configured.
gemini_voice_tool.run(sample_script)


---

## Step 4: Structured Research Output

We bring forward the typed research report and guardrail from Notebook 2 so downstream agents receive reliable context.


In [ ]:
class ResearchItem(BaseModel):
    title: str = Field(description="Short title for the finding")
    summary: str = Field(description="What happened or why it matters")
    listener_angle: str = Field(description="Why podcast listeners would care")
    source: str = Field(description="Source name or URL when available")


class ResearchReport(BaseModel):
    topic: str = Field(description="Podcast topic being researched")
    episode_angle: str = Field(description="Main angle for the episode")
    items: list[ResearchItem] = Field(description="Research findings for the episode")
    host_questions: list[str] = Field(description="Questions hosts can ask during the episode")


def validate_research_report(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if len(report.items) < 5:
        return (False, "The research report must include at least 5 findings.")
    if len(report.host_questions) < 3:
        return (False, "The report must include at least 3 host questions.")
    return (True, report)


---

## Step 5: The Full Voice-Enabled Crew

The scriptwriter gets the file tools and `gemini_voice_tool`. After writing the script, it calls the voice tool to generate the audio file.


In [ ]:
researcher = Agent(
    role="{topic} Senior Researcher",
    goal="Find recent releases, articles, topics, and progress about {topic}",
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=True,
    max_iter=5,
    max_rpm=12,
    max_execution_time=300,
    respect_context_window=True,
)

reporting_analyst = Agent(
    role="{topic} Reporting Analyst",
    goal="Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You are a meticulous analyst with a keen eye for detail. "
        "You turn complex data into clear podcast briefing reports."
    ),
    llm=llm,
    tools=[file_writer_tool, file_read_tool],
    verbose=True,
    max_iter=4,
    max_rpm=10,
    max_execution_time=240,
    respect_context_window=True,
)

scriptwriter = Agent(
    role="Podcast Scriptwriter",
    goal="Create a fun, natural, interesting podcast script given a report on {topic}",
    backstory=(
        "You are a naturally talented scriptwriter who turns technical reports into engaging, "
        "funny, and articulate podcast dialogue between two hosts."
    ),
    llm=llm,
    tools=[file_writer_tool, file_read_tool, gemini_voice_tool],
    verbose=True,
    max_iter=5,
    max_rpm=8,
    max_execution_time=420,
    respect_context_window=True,
)


In [ ]:
research_task = Task(
    description=(
        "Conduct thorough podcast research about {topic}. Find interesting and relevant information. "
        "The current date is {current_month} {current_year}."
    ),
    expected_output="A structured research report with at least 5 findings and 3 host questions.",
    output_pydantic=ResearchReport,
    guardrail=validate_research_report,
    agent=researcher,
)

reporting_task = Task(
    description=(
        "Review the research context and organize each topic into a full section for a report. "
        "Make the report detailed and useful for a podcast production team."
    ),
    expected_output="A markdown report with episode angle, sections, facts, host questions, and transitions.",
    guardrail=(
        "The report must include at least 3 clearly labeled sections with headers. "
        "Each section must contain at least 2 bullet points of useful facts."
    ),
    agent=reporting_analyst,
    context=[research_task],
    output_file="outputs/podcast_report.md",
)

# Prompt Engineering: Chain-of-Thought — sequential reasoning steps guide the agent before it writes
scripting_task = Task(
    description=(
        "Review the report and write an elaborate podcast script between two hosts. "
        "First, identify the 3-4 most compelling moments or surprising reveals — these become the episode peaks. "
        "Then plan a narrative arc: cold open hook, topic order, and a closing takeaway. "
        "Finally, write the full dialogue between Joe and Jane following that arc. "
        "The only speakers are Joe and Jane. Use speaker labels exactly as Joe: and Jane:. "
        "After writing the script, pass the full script to gemini_voice_tool to generate and save podcast audio."
    ),
    expected_output=(
        "A complete Joe/Jane podcast script and the path to the generated .wav audio file. "
        "Include sound effects cues, laughs, silences, and clear technical explanations."
    ),
    guardrail=(
        "The script must only use Joe and Jane as speakers. "
        "It must include at least 12 lines of dialogue and mention the generated audio file path."
    ),
    agent=scriptwriter,
    context=[reporting_task],
    output_file="outputs/podcast_script.md",
)


---

## Step 6: Run the End-to-End Podcaster

```text
Topic -> Researcher -> Reporting Analyst -> Scriptwriter -> Gemini Voice Tool -> WAV file
```


In [ ]:
voice_podcaster_crew = Crew(
    agents=[researcher, reporting_analyst, scriptwriter],
    tasks=[research_task, reporting_task, scripting_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=30,
)

inputs = {
    "topic": "AI LLMs",
    "current_month": datetime.datetime.now().strftime("%B"),
    "current_year": str(datetime.datetime.now().year),
}

final_result = voice_podcaster_crew.kickoff(inputs=inputs)
final_result


---

## Step 7: Reuse With Any Topic

Once the crew is defined, changing the episode topic is just changing the inputs.


In [ ]:
new_inputs = {
    "topic": "AI agents for software engineering",
    "current_month": datetime.datetime.now().strftime("%B"),
    "current_year": str(datetime.datetime.now().year),
}

# Uncomment to create another episode.
# voice_podcaster_crew.kickoff(inputs=new_inputs)


---

## Recap

| Step | You Learned | CrewAI API |
|------|-------------|------------|
| 1 | Register Python functions as tools | `@tool` |
| 2 | Give agents file system capabilities | `FileWriterTool`, `FileReadTool` |
| 3 | Build a custom TTS integration | `gemini_voice_tool(...)` |
| 4 | Wire tools into specialist agents | `tools=[...]` |
| 5 | Run the complete production flow | `Crew(...).kickoff(inputs=...)` |

You now have the full agent tutorial version of the personal podcaster: a multi-agent system that researches, writes, scripts, and generates audio — all from a single `kickoff()` call.
